#Download data set

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle
!kaggle datasets download -d francismon/curated-colon-dataset-for-deep-learning
!unzip -q curated-colon-dataset-for-deep-learning.zip

#Visualising Images

In [ ]:
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt
import cv2

def img_show(img_path, title):
    img = cv2.imread(img_path)[:,:,::-1]
    plt.imshow(img)
    plt.title(title)
    plt.show()

img_show('/content/train/0_normal/train_normal_ (1).jpg', 'Normal')

#DATA PREPROCESSING

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
trainPath = '/content/train'
testPath = '/content/test'
valPath = '/content/val'

img_size = (224, 224)
batch_size=16
# Instantiating ImageDataGenerator for train, test and validation sets
train_datagen = ImageDataGenerator(rescale=1./255,zoom_range=0.2,shear_range=0.2)
test_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)


train = train_datagen.flow_from_directory(trainPath,target_size =(224,224),batch_size=16,class_mode='categorical')
test = test_datagen.flow_from_directory(testPath,target_size =(224,224),batch_size=16,class_mode='categorical')
val = val_datagen.flow_from_directory(valPath,target_size =(224,224),batch_size=16,class_mode='categorical')

#Data Augumentation

In [ ]:
from tensorflow.keras import layers
import tensorflow as tf

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomHeight(0.2),
    layers.RandomWidth(0.2),
    layers.RandomContrast(0.2)
], name="data_augmentation")

# Test the augmentation layer
path = "/content/train/0_normal/train_normal_ (1).jpg"
img_show(path, "Image before augmenting")
img = cv2.imread(path)
augmented_img = data_augmentation(tf.expand_dims(img, axis=0))
augmented_img = tf.squeeze((augmented_img)/255.)[:, :, ::-1]

plt.figure()
plt.imshow(augmented_img)
plt.show()

#Inception

In [ ]:
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.layers import Dense,Flatten
from tensorflow.keras.models import Model

InceptionV3 = InceptionV3(include_top=False,input_shape=(224 ,224,3))

In [ ]:
for layer in InceptionV3.layers:
  print(layer)

In [ ]:
x = Flatten()(InceptionV3.output)
output = Dense(4,activation='softmax')(x)
InceptionV3 = Model(InceptionV3.input,output)
InceptionV3.summary()

In [ ]:
InceptionV3.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
InceptionV3.fit(train,validation_data=test,epochs=5)

import os

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
base_path = '/content'
model_save_path = os.path.join(base_path, "model/InceptionV3_model.h5")
##model.fit(train_generator, validation_data=validation_generator, epochs=2)
# Define callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint(model_save_path, monitor='val_loss', save_best_only=True, verbose=1)

# Compile the model
InceptionV3.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model with callbacks
history = InceptionV3.fit(
    train,
    validation_data=test,
    epochs=50,
    callbacks=[early_stopping, checkpoint]
)

# Print the model save path
print(f"Model saved at: {model_save_path}")

# Save the model
os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
InceptionV3.save(model_save_path)

In [ ]:
InceptionV3.save('InceptionV3.h5')
from google.colab import files
files.download("InceptionV3.h5")

#Load and Predict

In [ ]:
#load the model
from keras.models import load_model
model = load_model('/content/InceptionV3.h5')

#Predict
#import the library's
from keras.preprocessing import image
from keras.applications.vgg16 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np
index=['0_normal','1_ulcerative_colitis','2_polyps','3_esophagitis']
Img_path= r'/content/test/0_normal/test_normal_ (100).jpg'
img=image.load_img(Img_path,target_size=(224,224))
x=image.img_to_array(img)
x=np.expand_dims(x,axis=0) #(1,224,224,3)
img_data=preprocess_input(x)
img_data.shape
output=np.argmax(model.predict(img_data), axis=1)
result = index[output[0]]
result

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

testpath = os.path.join(base_path, "test")
test_datagen = ImageDataGenerator(rescale=1./255)
test = test_datagen.flow_from_directory(
    testpath,
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical',
    shuffle=False
)

# Plotting loss and accuracy curves
def plot_loss_curves(history):
    # Loss Curves
    plt.figure(figsize=(8, 5))
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Loss Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

    # Accuracy Curves
    plt.figure(figsize=(8, 5))
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Accuracy Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.show()

plot_loss_curves(history)

# Model evaluation on the test set
test_loss, test_accuracy = model.evaluate(test)
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

# Classification report and confusion matrix
y_true = test.classes
y_pred = np.argmax(model.predict(test), axis=-1)

# Extracting class names from the data generator
class_names = list(train.class_indices.keys())

print('Classification Report')
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

print('Confusion Matrix:')
print(cm)